In [ ]:
from pathlib import Path

# Root van team_1
BASE_DIR = Path.cwd().parent

# Mappen
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"

print(DATA_DIR)
print(OUTPUT_DIR)


In [ ]:
gdb_2011 = DATA_DIR / "LeggerRegionalekeringzone2011.gdb"
gdb_2013 = DATA_DIR / "LeggerRegionalekering2013.gdb"
gdb_vig  = DATA_DIR / "LeggerRegionalekeringVigerend.gdb"
gdb_rijn = DATA_DIR / "LeggerRegionalekeringRijnland.gdb"
gdb_watergangen = DATA_DIR / "Watergang.gdb"


In [ ]:
import geopandas as gpd
from pathlib import Path

# Buitenste map
outer = Path(
    r"C:\Users\bvendrig\OneDrive - Waterschap\Documenten\VScode\Geo\LeggerRegionalekeringRijnland.gdb"
)

# Zoek alle gdb's hieronder
for gdb in outer.rglob("*.gdb"):
    print("\nGDB gevonden:")
    print(gdb)

    try:
        layers = gpd.list_layers(str(gdb))

        print("\nBeschikbare lagen:")
        print(layers)

    except Exception as e:
        print(f"Fout: {e}")

In [ ]:

import geopandas as gpd
import pandas as pd

gdb_path = (
    r"C:\Users\bvendrig\OneDrive - Waterschap\Documenten\VScode\Geo"
    r"\LeggerRegionalekeringRijnland.gdb\LeggerRegionalekeringRijnland.gdb"
)

laag = "Regionalekeringzonevigerend"

print("\n" + "=" * 100)
print(f"LAAG: {laag}")
print("=" * 100)

gdf = gpd.read_file(gdb_path, layer=laag)

print(f"Aantal records: {len(gdf):,}")
print(f"Aantal kolommen: {len(gdf.columns)}")

overzicht = pd.DataFrame({
    "Kolom": gdf.columns,
    "Datatype": [str(dtype) for dtype in gdf.dtypes]
})

print("\nKolommen:")
print(overzicht.to_string(index=False))

print("\nEerste record:")
print(gdf.iloc[0].drop("geometry", errors="ignore"))

print("\nUnieke waarden per kolom (max 20):")

for col in gdf.columns:

    if col == "geometry":
        continue

    try:
        waarden = (
            gdf[col]
            .dropna()
            .astype(str)
            .unique()
        )

        if len(waarden) <= 20:
            print(f"\n{col}")
            print(waarden)

    except Exception:
        pass

In [ ]:
import geopandas as gpd

gdb_path = (
    r"C:\Users\bvendrig\OneDrive - Waterschap\Documenten\VScode\Geo"
    r"\LeggerRegionalekeringRijnland.gdb\LeggerRegionalekeringRijnland.gdb"
)

gdf = gpd.read_file(
    gdb_path,
    layer="Regionalekeringzonevigerend"
)

print(
    sorted(
        gdf["TYPEZONE"]
        .dropna()
        .unique()
    )
)

In [ ]:
from pathlib import Path
import geopandas as gpd

gdb_2013 = r"C:\Users\bvendrig\OneDrive - Waterschap\Documenten\VScode\Geo\LeggerRegionalekeringzone2013.gdb"
gdb_vigerend = r"C:\Users\bvendrig\OneDrive - Waterschap\Documenten\VScode\Geo\LeggerRegionalekeringVigerend.gdb"
gdb_2013 = r"C:\Users\bvendrig\OneDrive - Waterschap\Documenten\VScode\Geo\LeggerRegionalekering2013.gdb"

print("Bestaat 20133:", Path(gdb_2013).exists())
print("Bestaat 2013:", Path(gdb_2013).exists())
print("Bestaat vigerend:", Path(gdb_vigerend).exists())

print("\nInhoud 2013:")
for item in Path(gdb_2013).iterdir():
    print(item.name)
    break

print("\nInhoud vigerend:")
for item in Path(gdb_vigerend).iterdir():
    print(item.name)
    break

In [ ]:
import pyogrio

print(pyogrio.read_info(gdb_2013))


In [ ]:
import geopandas as gpd
import folium
from shapely.geometry import box

# --------------------------------------------------
# GDB bestanden
# --------------------------------------------------

gdb_2013 = DATA_DIR / "LeggerRegionalekering2013.gdb"
gdb_vigerend = DATA_DIR / "LeggerRegionalekeringVigerend.gdb"


# --------------------------------------------------
# Inlezen
# --------------------------------------------------

zone2013 = gpd.read_file(
    gdb_2013,
    layer="LeggerRegionalekeringzone2013"
)

zonevig = gpd.read_file(
    gdb_vigerend,
    layer="Regionalekeringzonevigerend"
)

# --------------------------------------------------
# Watergangen
# --------------------------------------------------

watergang_gdb = DATA_DIR / "Watergang.gdb"

watergang = gpd.read_file(
watergang_gdb,
layer="WatergangVlak"
)

# --------------------------------------------------
# Alleen kernzones
# --------------------------------------------------

zone2013 = zone2013[
    zone2013["TYPEZONE"] == "kernzone waterkering"
].copy()

zonevig = zonevig[
    zonevig["TYPEZONE"] == "Kernzone"
].copy()

# --------------------------------------------------
# Verschillen bepalen
# --------------------------------------------------

toegevoegd = gpd.overlay(
    zonevig,
    zone2013,
    how="difference"
)

verdwenen = gpd.overlay(
    zone2013,
    zonevig,
    how="difference"
)

# --------------------------------------------------
# Kleine geometrische verschillen negeren
# --------------------------------------------------

# min_oppervlakte = 25

# toegevoegd = toegevoegd[
#     toegevoegd.area > min_oppervlakte
# ].copy()

# verdwenen = verdwenen[
#     verdwenen.area > min_oppervlakte
# ].copy()

# --------------------------------------------------
# Kenmerken toevoegen
# --------------------------------------------------

def voeg_kenmerken_toe(gdf, wijziging):

    if len(gdf) == 0:
        return gdf

    gdf = gdf.copy()

    gdf["Wijziging"] = wijziging
    gdf["Oppervlakte_m2"] = gdf.area.round(1)
    gdf["Omtrek_m"] = gdf.length.round(1)

    # Nieuwe maat
    gdf["Gem_Breedte_m"] = (
        2 * gdf.area / gdf.length
    ).round(2)

    breedtes = []
    lengtes = []

    for geom in gdf.geometry:

        try:

            rect = geom.minimum_rotated_rectangle

            coords = list(rect.exterior.coords)

            zijdes = []

            for i in range(4):

                x1, y1 = coords[i]
                x2, y2 = coords[i + 1]

                zijde = (
                    ((x2 - x1) ** 2) +
                    ((y2 - y1) ** 2)
                ) ** 0.5

                zijdes.append(zijde)

            zijdes = sorted(zijdes)

            breedtes.append(round(zijdes[0], 1))
            lengtes.append(round(zijdes[-1], 1))

        except:

            breedtes.append(None)
            lengtes.append(None)

    gdf["Breedte_m"] = breedtes
    gdf["Lengte_m"] = lengtes

    return gdf

toegevoegd = voeg_kenmerken_toe(
    toegevoegd,
    "Toegevoegd sinds 2013"
)

verdwenen = voeg_kenmerken_toe(
    verdwenen,
    "Verdwenen sinds 2013"
)

# --------------------------------------------------
# WGS84 voor Folium
# --------------------------------------------------

zone2013_wgs = zone2013.to_crs(4326)
zonevig_wgs = zonevig.to_crs(4326)

toegevoegd_wgs = toegevoegd.to_crs(4326)
verdwenen_wgs = verdwenen.to_crs(4326)

watergang_wgs = watergang.to_crs(4326)
# --------------------------------------------------
# Kaart maken
# --------------------------------------------------

bounds = zonevig_wgs.total_bounds

lat = (bounds[1] + bounds[3]) / 2
lon = (bounds[0] + bounds[2]) / 2

m = folium.Map(
    location=[lat, lon],
    zoom_start=14,
    tiles=None
)

# --------------------------------------------------
# Achtergrondlagen
# --------------------------------------------------

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Imagery",
    name="Luchtfoto",
    overlay=False
).add_to(m)


# --------------------------------------------------
# Historische kernzone
# --------------------------------------------------

zone2013_wgs.explore(
    m=m,
    name="Kernzone 2013",
    show=False,
    color="orange",
    style_kwds={
        "fillOpacity": 0.05,
        "weight": 2
    }
)

# --------------------------------------------------
# Huidige kernzone
# --------------------------------------------------

zonevig_wgs.explore(
    m=m,
    name="Kernzone vigerend",
    show=False,
    color="blue",
    style_kwds={
        "fillOpacity": 0.05,
        "weight": 2
    }
)

# --------------------------------------------------
# Toegevoegd
# --------------------------------------------------

if len(toegevoegd_wgs) > 0:

    toegevoegd_wgs.explore(
        m=m,
        name="Toegevoegd sinds 2013",
        color="lime",
        tooltip=[
            "Wijziging",
            "Oppervlakte_m2",
            "Gem_Breedte_m",
            # "Breedte_m",
            # "Lengte_m",
            # "Omtrek_m"
        ],
        style_kwds={
            "fillOpacity": 0.8,
            "weight": 1
        }
    )

# --------------------------------------------------
# Verdwenen
# --------------------------------------------------

if len(verdwenen_wgs) > 0:

    verdwenen_wgs.explore(
        m=m,
        name="Verdwenen sinds 2013",
        color="red",
        tooltip=[
            "Wijziging",
            "Oppervlakte_m2",
            "Gem_Breedte_m",
            # "Breedte_m",
            # "Lengte_m",
            # "Omtrek_m"
        ],
        style_kwds={
            "fillOpacity": 0.8,
            "weight": 1
        }
    )

# --------------------------------------------------
# Layer control
# --------------------------------------------------

legend_html = """
<div style="
position: fixed;
bottom: 50px;
left: 50px;
width: 260px;
background-color: white;
border:2px solid grey;
z-index:9999;
font-size:14px;
padding: 10px;
">

<b>Leggervergelijking 2013 vs Vigerend</b><br><br>

<span style="
display:inline-block;
width:20px;
height:12px;
background:lime;
border:1px solid black;
"></span>
Toegevoegd sinds 2013
<br><br>

<span style="
display:inline-block;
width:20px;
height:12px;
background:red;
border:1px solid black;
"></span>
Verdwenen sinds 2013
<br><br>

<span style="
display:inline-block;
width:20px;
height:12px;
background:orange;
border:1px solid black;
"></span>
Kernzone 2013
<br><br>

<span style="
display:inline-block;
width:20px;
height:12px;
background:blue;
border:1px solid black;
"></span>
Kernzone vigerend

<br><br>

<span style="
display:inline-block;
width:20px;
height:12px;
background:#00BFFF;
border:1px solid black;
"></span>
Huidige watergangen

</div>
"""

m.get_root().html.add_child(
    folium.Element(legend_html)
)

# --------------------------------------------------
# Huidige watergangen
# --------------------------------------------------

# watergang_wgs.explore(
#     m=m,
#     name="Huidige watergangen",
#     show=False,
#     color="#00BFFF",
#     style_kwds={
#         "fillOpacity": 0.25,
#         "weight": 0.5
#     }
# )

folium.LayerControl(
    collapsed=False
).add_to(m)

m.fit_bounds([
    [bounds[1], bounds[0]],
    [bounds[3], bounds[2]]
])


# --------------------------------------------------
# Resultaten exporteren voor Power BI
# --------------------------------------------------

from pathlib import Path
import pandas as pd

output_map = OUTPUT_DIR / "Gouda_Kernzones_2013_vs_Vigerend.html"
output_csv = OUTPUT_DIR / "Gouda_Kernzones_2013_vs_Vigerend.csv"
output_geojson = OUTPUT_DIR / "Gouda_Kernzones_2013_vs_Vigerend.geojson"

# --------------------------------------------------
# Exportfunctie
# --------------------------------------------------

def maak_exporttabel(gdf, wijziging_code):

    if gdf.empty:
        return gpd.GeoDataFrame(
            columns=[
                "Wijziging_ID",
                "Vergelijkingsjaar",
                "Referentie",
                "Wijziging",
                "Oppervlakte_m2",
                "Gem_Breedte_m",
                "Breedte_m",
                "Lengte_m",
                "Omtrek_m",
                "Centroid_X_RD",
                "Centroid_Y_RD",
                "Centroid_Latitude",
                "Centroid_Longitude",
                "Geometry_WKT",
                "geometry"
            ],
            geometry="geometry",
            crs=gdf.crs
        )

    export_gdf = gdf.copy().reset_index(drop=True)

    # Uniek nummer per verschilvlak
    export_gdf["Wijziging_ID"] = [
        f"{wijziging_code}-{nummer:05d}"
        for nummer in range(1, len(export_gdf) + 1)
    ]

    export_gdf["Vergelijkingsjaar"] = 2013
    export_gdf["Referentie"] = "Vigerend"
    export_gdf["JaarVergelijking"] = "2013-vigerend"

    # Centroid in het oorspronkelijke projectie-CRS
    centroid_rd = export_gdf.geometry.centroid

    export_gdf["Centroid_X_RD"] = centroid_rd.x.round(2)
    export_gdf["Centroid_Y_RD"] = centroid_rd.y.round(2)

    # Centroid omzetten naar WGS84 voor Power BI-kaarten
    centroid_wgs84 = gpd.GeoSeries(
        centroid_rd,
        crs=export_gdf.crs
    ).to_crs(4326)

    export_gdf["Centroid_Latitude"] = centroid_wgs84.y.round(7)
    export_gdf["Centroid_Longitude"] = centroid_wgs84.x.round(7)

    # Polygon als tekst, bruikbaar voor visuals die WKT accepteren
    export_gdf["Geometry_WKT"] = export_gdf.geometry.to_wkt(
        rounding_precision=2
    )

    gewenste_kolommen = [
        "Wijziging_ID",
        "Vergelijkingsjaar",
        "Referentie",
        "JaarVergelijking",
        "Wijziging",
        "Oppervlakte_m2",
        "Gem_Breedte_m",
        "Breedte_m",
        "Lengte_m",
        "Omtrek_m",
        "Centroid_X_RD",
        "Centroid_Y_RD",
        "Centroid_Latitude",
        "Centroid_Longitude",
        "Geometry_WKT",
        "geometry"
    ]

    return export_gdf[gewenste_kolommen]


# --------------------------------------------------
# Toegevoegd en verdwenen combineren
# --------------------------------------------------

export_toegevoegd = maak_exporttabel(
    toegevoegd,
    "TOE"
)

export_verdwenen = maak_exporttabel(
    verdwenen,
    "VER"
)

export_geo = gpd.GeoDataFrame(
    pd.concat(
        [export_toegevoegd, export_verdwenen],
        ignore_index=True
    ),
    geometry="geometry",
    crs=toegevoegd.crs
)

# --------------------------------------------------
# CSV voor Power BI
#
# De geometriekolom zelf wordt verwijderd omdat deze
# niet rechtstreeks als Shapely-object naar CSV kan.
# Geometry_WKT blijft wel behouden.
# --------------------------------------------------

export_tabel = pd.DataFrame(
    export_geo.drop(columns="geometry")
)

export_tabel.to_csv(
    output_csv,
    index=False,
    encoding="utf-8-sig",
    sep=";"
)

# --------------------------------------------------
# GeoJSON met geometrie
# --------------------------------------------------

if not export_geo.empty:

    export_geo_wgs84 = export_geo.to_crs(4326)

    export_geo_wgs84.to_file(
        output_geojson,
        driver="GeoJSON"
    )

# --------------------------------------------------
# Interactieve HTML-kaart opslaan
# --------------------------------------------------

m.save(output_map)

# --------------------------------------------------
# Exportcontrole
# --------------------------------------------------

print("\nExport afgerond")
print("-" * 60)
print(f"HTML-kaart : {output_map}")
print(f"Power BI CSV: {output_csv}")
print(f"GeoJSON     : {output_geojson}")
print(f"Rijen totaal: {len(export_tabel):,}")
print(f"Toegevoegd  : {len(export_toegevoegd):,}")
print(f"Verdwenen   : {len(export_verdwenen):,}")
print()
print(export_tabel.head())


m